# XGB rolling cutoff training + prediction + submission generation

This notebook is isolated from `xgb_train_predict.ipynb` and focuses on:
- Concatenating multiple `cutoff_time` snapshots for training (rolling / sliding cutoff)
- Labels: `label_mode="horizon"` + `horizon_days=10` (churn within the next 10 days)
- Submission: build test features with a fixed cutoff (usually the last day, e.g. 2018-11-10) and output `id,target`

Tip: if cutoffs are too dense, the sample size becomes `num_users × num_cutoffs`. Start with a coarse stride (2D/3D/5D), then refine.


In [ ]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from xgboost import XGBClassifier

ROOT = Path().resolve()
base_path = '/Data/yuhan.wu/kaggle_project_new'
sys.path.append(f'{base_path}/src')
from churn_pipeline.dataset_builder import build_datasets_cli  # noqa: E402

## 1) Configuration: cutoffs, label, feature window, XGB params

In [ ]:
# Data paths
COMPETITION_DIR = Path(f"{base_path}/churn-prediction-25-26")
TRAIN_PATH = COMPETITION_DIR / "train.parquet"
TEST_PATH = COMPETITION_DIR / "test.parquet"
EXAMPLE_SUB = COMPETITION_DIR / "example_submission.csv"

# rolling cutoffs（先稀疏跑通，再加密）
CUTOFF_START = "2018-10-20"
CUTOFF_END = "2018-11-10"
CUTOFF_FREQ = "2D"  # "1D" | "2D" | "3D" | ...
CUT_OFFS = pd.date_range(CUTOFF_START, CUTOFF_END, freq=CUTOFF_FREQ)

# 标签
LABEL_MODE = "horizon"
HORIZON_DAYS = 10

# 特征窗口：看最近多少天
LOOKBACK_DAYS = 51

# 防泄露过滤（与主 notebook 对齐）
EXCLUDE_PAGES = "Cancellation Confirmation,Cancel"
EXCLUDE_AUTH_VALUES = "Cancelled"

# build 参数
DROP_LAST_K_EVENTS = 0
FULL_CALENDAR = True
VAL_PROB_LOW = 0.8
VAL_PROB_HIGH = 1.0
SPLIT_SEED = "split_v1"
CACHE_DIR = ".cache/user_day_features"
OUTPUT_BASE_DIR = "data/processed"
BATCH_SIZE = 1_000_000
FORCE_RECOMPUTE = False

def bal_acc(y_true, y_pred) -> float:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_hat = (y_pred >= 0.5).astype(int)
    return float(balanced_accuracy_score(y_true, y_hat))

# XGB 参数
XGB_DEVICE = "cuda"  # "cuda" | "cpu"
xgb_params = dict(
    n_estimators=5000,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=2.0,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_lambda=0.5,
    reg_alpha=0.5,
    gamma=0.0,
    objective="binary:logistic",
    # eval_metric=bal_acc,
    eval_metric='auc',
    tree_method="hist",
    device=XGB_DEVICE,
    random_state=42,
    n_jobs=-1,

)

EARLY_STOPPING_ROUNDS = 200
XGB_CALLBACKS = [
    xgb.callback.EarlyStopping(
        rounds=int(EARLY_STOPPING_ROUNDS),
        # metric_name="bal_acc",
        metric_name="auc",
        data_name="validation_0",
        maximize=True,
        save_best=False,
    )
]

print("cutoffs=", len(CUT_OFFS), CUT_OFFS.min().date(), "->", CUT_OFFS.max().date())

## 2) Utilities: expand user-day into fixed-length per-user features

Flatten in memory (no intermediate parquet files).


In [ ]:
def _to_day(value: str | pd.Timestamp | np.datetime64) -> np.datetime64:
    if isinstance(value, np.datetime64):
        return value.astype("datetime64[D]")
    return np.datetime64(pd.Timestamp(value).normalize().to_datetime64(), "D")


def build_window_features_from_df(
    user_day: pd.DataFrame,
    *,
    lookback_days: int,
    end_day: str | pd.Timestamp,
    fill_value: float = 0.0,
    dtype: str = "float32",
) -> pd.DataFrame:
    lookback = int(lookback_days)
    if lookback <= 0:
        raise ValueError(f"lookback_days must be > 0, got {lookback}")

    required = {"userId", "day"}
    missing = required - set(user_day.columns)
    if missing:
        raise ValueError(f"user_day missing required columns: {sorted(missing)}")

    user_day = user_day.sort_values(["userId", "day"], kind="mergesort")

    end_d = _to_day(end_day)
    start_d = end_d - np.timedelta64(lookback - 1, "D")

    day_d = pd.to_datetime(user_day["day"]).values.astype("datetime64[D]")
    in_window = (day_d >= start_d) & (day_d <= end_d)
    user_day = user_day.loc[in_window].copy()
    if user_day.empty:
        raise ValueError("No rows left after applying the window; check end_day/lookback_days.")

    lag = (end_d - day_d[in_window]).astype(int)
    user_day["lag"] = lag.astype(np.int16)

    feature_cols = [c for c in user_day.columns if c not in {"userId", "day", "lag"}]
    if not feature_cols:
        raise ValueError("No feature columns found (expected columns besides userId/day).")

    wide = user_day.pivot(index="userId", columns="lag", values=feature_cols)
    full_cols = pd.MultiIndex.from_product([feature_cols, range(lookback)], names=["feature", "lag"])
    wide = wide.reindex(columns=full_cols)

    pad = len(str(lookback - 1))
    wide.columns = [f"{feat}_t-{int(l):0{pad}d}" for feat, l in wide.columns]

    wide = wide.fillna(float(fill_value))
    wide = wide.sort_index()
    if dtype:
        wide = wide.astype(dtype, copy=False)
    return wide

## 3) Build rolling train/val (concatenate multiple cutoff snapshots)

Note: the first run can be slow because features are generated and cached.


In [4]:
from tqdm import tqdm


t0 = time.time()

all_X_train = []
all_y_train = []
all_X_val = []
all_y_val = []

for cutoff in tqdm(CUT_OFFS):
    cutoff_str = str(pd.Timestamp(cutoff).date())
    paths = build_datasets_cli(
        train_path=str(TRAIN_PATH),
        test_path=str(TEST_PATH),
        output_dir=None,
        output_base_dir=str(OUTPUT_BASE_DIR),
        cache_dir=str(CACHE_DIR),
        cutoff_time=cutoff_str,
        horizon_days=int(HORIZON_DAYS),
        label_mode=str(LABEL_MODE),
        drop_last_k_events=int(DROP_LAST_K_EVENTS),
        exclude_pages=str(EXCLUDE_PAGES),
        exclude_auth_values=str(EXCLUDE_AUTH_VALUES),
        val_prob_low=float(VAL_PROB_LOW),
        val_prob_high=float(VAL_PROB_HIGH),
        split_seed=str(SPLIT_SEED),
        batch_size=int(BATCH_SIZE),
        full_calendar=bool(FULL_CALENDAR),
        force_recompute=bool(FORCE_RECOMPUTE),
        verbose=False,
    )

    train_ud = pd.read_parquet(paths["train_features"])
    val_ud = pd.read_parquet(paths["val_features"])
    train_lab = pd.read_parquet(paths["train_labels"])[["userId", "label"]]
    val_lab = pd.read_parquet(paths["val_labels"])[["userId", "label"]]

    end_day = pd.Timestamp(cutoff).normalize() - pd.Timedelta(days=1)
    start_day = end_day - pd.Timedelta(days=int(LOOKBACK_DAYS) - 1)

    def _slice(df: pd.DataFrame) -> pd.DataFrame:
        d = pd.to_datetime(df["day"]).dt.normalize()
        return df.loc[(d >= start_day) & (d <= end_day)].copy()

    train_ud = _slice(train_ud)
    val_ud = _slice(val_ud)

    suffix = "|" + cutoff_str
    train_ud["userId"] = train_ud["userId"].astype(str) + suffix
    val_ud["userId"] = val_ud["userId"].astype(str) + suffix
    train_lab["userId"] = train_lab["userId"].astype(str) + suffix
    val_lab["userId"] = val_lab["userId"].astype(str) + suffix

    X_tr = build_window_features_from_df(train_ud, lookback_days=int(LOOKBACK_DAYS), end_day=end_day)
    X_va = build_window_features_from_df(val_ud, lookback_days=int(LOOKBACK_DAYS), end_day=end_day)

    y_tr = train_lab.set_index("userId")["label"].reindex(X_tr.index).astype("float")
    y_va = val_lab.set_index("userId")["label"].reindex(X_va.index).astype("float")

    keep_tr = y_tr.notna()
    keep_va = y_va.notna()
    X_tr, y_tr = X_tr.loc[keep_tr], y_tr.loc[keep_tr].astype(int)
    X_va, y_va = X_va.loc[keep_va], y_va.loc[keep_va].astype(int)

    all_X_train.append(X_tr)
    all_y_train.append(y_tr)
    all_X_val.append(X_va)
    all_y_val.append(y_va)

X_train = pd.concat(all_X_train, axis=0)
y_train = pd.concat(all_y_train, axis=0)
X_val = pd.concat(all_X_val, axis=0)
y_val = pd.concat(all_y_val, axis=0)

X_train, X_val = X_train.align(X_val, join="outer", axis=1, fill_value=0.0)

print("train", X_train.shape, "val", X_val.shape)
print("seconds", round(time.time() - t0, 2))

100%|██████████| 11/11 [08:35<00:00, 46.85s/it]


train (168003, 5253) val (42537, 5253)
seconds 516.52


In [5]:
# sanity check: 新增特征是否进入了最终 X_train/X_val（会被展开成 *_t-xx 列）
expected_new = [
    "u_tenure_days",
    "d_song_bucket_unique",
    "d_artist_bucket_unique",
    "d_song_repeat_rate",
    "d_artist_repeat_rate",
    "x_roll7_events_sum",
    "x_roll7_events_mean",
    "x_roll7_nextsong_sum",
    "x_roll7_nextsong_mean",
    "x_roll7_active_days",
    "x_trend7_events",
    "x_trend7_nextsong",
    "x_days_since_last_active",
    "x_active_streak",
]

def _has_prefix(cols, prefix: str) -> bool:
    p = prefix + "_t-"
    return any(str(c).startswith(p) for c in cols)

missing_prefix = [c for c in expected_new if not _has_prefix(X_train.columns, c)]
print("missing prefixes in X_train:", missing_prefix)
print("train features:", X_train.shape[1], "val features:", X_val.shape[1])

missing prefixes in X_train: []
train features: 5253 val features: 5253


## 4) Train XGB + validate AUC

In [6]:
pos = int((y_train == 1).sum())
neg = int((y_train == 0).sum())
# scale_pos_weight = float(np.sqrt(neg / max(pos, 1)))
scale_pos_weight = 5
print("pos/neg", pos, neg, "scale_pos_weight", scale_pos_weight)

train_params = dict(xgb_params)
train_params["scale_pos_weight"] = scale_pos_weight

model = XGBClassifier(**train_params, callbacks=XGB_CALLBACKS)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)

val_proba = model.predict_proba(X_val)[:, 1]
val_auc = float(roc_auc_score(y_val, val_proba))
val_auc

pos/neg 6788 161215 scale_pos_weight 5
[0]	validation_0-auc:0.71391
[100]	validation_0-auc:0.74557
[200]	validation_0-auc:0.74930
[300]	validation_0-auc:0.74994
[400]	validation_0-auc:0.74845
[478]	validation_0-auc:0.74745


/Data/yuhan.wu/miniconda3/envs/py310/lib/python3.10/site-packages/xgboost/core.py:774: UserWarning: [15:56:57] WARNING: /workspace/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


0.7502421222133668

## 5) 选阈值（Balanced Accuracy）

In [7]:
thresholds = np.linspace(0.01, 0.99, 99)
bal_scores = [balanced_accuracy_score(y_val, (val_proba >= t).astype(int)) for t in thresholds]
best_idx = int(np.argmax(bal_scores))
best_thr = float(thresholds[best_idx])
best_bal = float(bal_scores[best_idx])
best_thr, best_bal

(0.18000000000000002, 0.6867152736699667)

## 6) Train final model (train+val)

In [8]:
best_iteration = getattr(model, "best_iteration", None)
best_n_estimators = int(model.get_params()["n_estimators"]) if best_iteration is None else int(best_iteration) + 1
best_n_estimators

X_all = pd.concat([X_train, X_val], axis=0)
y_all = pd.concat([y_train, y_val], axis=0)

final_params = dict(train_params)
final_params["n_estimators"] = best_n_estimators

final_model = XGBClassifier(**final_params)
final_model.fit(X_all, y_all, verbose=False)
(X_all.shape, y_all.shape)

((210540, 5253), (210540,))

## 7) Generate test submission with a fixed cutoff

In [9]:
SUBMISSION_CUTOFF = str(pd.Timestamp(CUTOFF_END).date())

sub_paths = build_datasets_cli(
    train_path=str(TRAIN_PATH),
    test_path=str(TEST_PATH),
    output_dir=None,
    output_base_dir=str(OUTPUT_BASE_DIR),
    cache_dir=str(CACHE_DIR),
    cutoff_time=SUBMISSION_CUTOFF,
    horizon_days=int(HORIZON_DAYS),
    label_mode=str(LABEL_MODE),
    drop_last_k_events=int(DROP_LAST_K_EVENTS),
    exclude_pages=str(EXCLUDE_PAGES),
    exclude_auth_values=str(EXCLUDE_AUTH_VALUES),
    val_prob_low=float(VAL_PROB_LOW),
    val_prob_high=float(VAL_PROB_HIGH),
    split_seed=str(SPLIT_SEED),
    batch_size=int(BATCH_SIZE),
    full_calendar=bool(FULL_CALENDAR),
    force_recompute=bool(FORCE_RECOMPUTE),
    verbose=False,
)

test_ud = pd.read_parquet(sub_paths["test_features"])
cutoff_ts = pd.Timestamp(SUBMISSION_CUTOFF).normalize()
end_day = cutoff_ts - pd.Timedelta(days=1)
start_day = end_day - pd.Timedelta(days=int(LOOKBACK_DAYS) - 1)
d = pd.to_datetime(test_ud["day"]).dt.normalize()
test_ud = test_ud.loc[(d >= start_day) & (d <= end_day)].copy()

X_test = build_window_features_from_df(test_ud, lookback_days=int(LOOKBACK_DAYS), end_day=end_day)
X_test = X_test.reindex(columns=X_all.columns, fill_value=0.0)

test_proba = final_model.predict_proba(X_test)[:, 1]
thr = best_thr  # 或者改成 0.5

pred = pd.DataFrame({"id": X_test.index.astype(int), "proba": test_proba.astype(float)})
example = pd.read_csv(EXAMPLE_SUB)
out = example[["id"]].merge(pred, on="id", how="left")
out["proba"] = out["proba"].fillna(0.0)
out["target"] = (out["proba"] >= float(thr)).astype(int)
out = out[["id", "target"]]

print("test predicted positive rate:", float(out["target"].mean()))
out.head()

test predicted positive rate: 0.41632231404958675


,id,target
0,1128274,0
1,1782451,0
2,1611542,1
3,1241663,1
4,1653104,1


## 8) Save artifacts and submission files (experiments/xgb_rolling/)

In [ ]:
run_name = f"xgb_rolling_nb_cut{CUTOFF_START}-{CUTOFF_END}_{CUTOFF_FREQ}_sub{SUBMISSION_CUTOFF}_h{HORIZON_DAYS}_lb{LOOKBACK_DAYS}"

ART_DIR = ROOT / "experiments" / "xgb_rolling" / "artifacts" / run_name
SUB_DIR = ROOT / "experiments" / "xgb_rolling" / "submissions"
ART_DIR.mkdir(parents=True, exist_ok=True)
SUB_DIR.mkdir(parents=True, exist_ok=True)

model_path = ART_DIR / "model.json"
final_model.save_model(model_path)

(ART_DIR / "feature_columns.json").write_text(json.dumps(list(X_all.columns), ensure_ascii=False, indent=2), encoding="utf-8")

meta = {
    "run_name": run_name,
    "cutoffs": [str(pd.Timestamp(c).date()) for c in CUT_OFFS],
    "submission_cutoff": SUBMISSION_CUTOFF,
    "lookback_days": int(LOOKBACK_DAYS),
    "horizon_days": int(HORIZON_DAYS),
    "label_mode": str(LABEL_MODE),
    "scale_pos_weight": float(scale_pos_weight),
    "train_rows": int(X_train.shape[0]),
    "val_rows": int(X_val.shape[0]),
    "num_features": int(X_all.shape[1]),
    "val_auc": float(val_auc),
    "best_n_estimators": int(best_n_estimators),
    "best_thr_bal": float(best_thr),
    "best_bal_acc": float(best_bal),
    "xgb_params": train_params,
    "final_params": final_params,
}
# (ART_DIR / "train_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")

sub_path = SUB_DIR / f"{run_name}.csv"
out.to_csv(sub_path, index=False)

model_path, sub_path

(PosixPath('/Data/yuhan.wu/kaggle_project_new/experiments/xgb_rolling/experiments/xgb_rolling/artifacts/xgb_rolling_nb_cut2018-10-20-2018-11-10_2D_sub2018-11-10_h10_lb51/model.json'),
 PosixPath('/Data/yuhan.wu/kaggle_project_new/experiments/xgb_rolling/experiments/xgb_rolling/submissions/xgb_rolling_nb_cut2018-10-20-2018-11-10_2D_sub2018-11-10_h10_lb51.csv'))

In [11]:
# Optional: submit to Kaggle (requires ~/.kaggle/kaggle.json)
import sys
sys.path.append(f"{base_path}/src")
sys.path.append(base_path)
from kaggle_submit import submit_and_track

competition = "churn-prediction-25-26"
message = f"xgb slide "
info = submit_and_track(
    sub_path,
    competition=competition,
    message=message,
)
info


Submitting /Data/yuhan.wu/kaggle_project_new/experiments/xgb_rolling/experiments/xgb_rolling/submissions/xgb_rolling_nb_cut2018-10-20-2018-11-10_2D_sub2018-11-10_h10_lb51.csv to churn-prediction-25-26 with note: xgb slide 


100%|██████████| 28.4k/28.4k [00:00<00:00, 62.8kB/s]


Poll 1/8: status=complete score=0.61509


{'ref': 49059987,
 'description': 'xgb slide ',
 'status': 'complete',
 'score': 0.61509,
 'fileName': 'xgb_rolling_nb_cut2018-10-20-2018-11-10_2D_sub2018-11-10_h10_lb51.csv',
 'submissionDate': datetime.datetime(2025, 12, 14, 14, 58, 15, 177000),
 'error': '',
 'teamName': 'Yuhan Wu',
 'submittedBy': 'yuhanwu1003',
 'isFrozen': True}

## 9) Heuristic tuning (coordinate search; optimize local val Balanced Accuracy)

Idea: tune one hyperparameter at a time while keeping the others fixed. For each candidate value, evaluate and pick the best (primary: `val_bal_acc`, secondary: `val_auc`), then move on to the next hyperparameter. Repeat for multiple rounds until no further improvement (local optimum).

Outputs:
- Append every evaluation to a CSV log (includes `val_bal_acc` / `thr` / `run_name` / `p_*` columns).
- Also export a Top-3 CSV (handy for choosing what to upload to Kaggle).
- Update `experiments/best_params.json`: record Top-3 in `xgb_rolling.top_runs`, and set `xgb_rolling.run_name` to Top-1 (the ensemble reads it by default).


In [12]:
# import csv
# import hashlib
# import json
# from datetime import datetime
# from pathlib import Path

# import numpy as np
# import pandas as pd
# from sklearn.metrics import balanced_accuracy_score, roc_auc_score
# from xgboost import XGBClassifier

# # 兼容：如果你只单独跑本 cell（没从头执行 notebook），先兜底 ROOT
# try:
#     ROOT  # type: ignore[name-defined]
# except NameError:
#     ROOT = Path().resolve()

# _required_names = [
#     "xgb_params",
#     "XGB_CALLBACKS",
#     "X_train",
#     "y_train",
#     "X_val",
#     "y_val",
#     "X_test",
#     "EXAMPLE_SUB",
#     "CUTOFF_START",
#     "CUTOFF_END",
#     "CUTOFF_FREQ",
#     "SUBMISSION_CUTOFF",
#     "HORIZON_DAYS",
#     "LOOKBACK_DAYS",
# ]
# _missing = [n for n in _required_names if n not in globals()]
# if _missing:
#     raise RuntimeError(
#         "Missing variables from earlier cells: " + ", ".join(_missing) + ". "
#         "Please run the previous cells (1-8) to build datasets/train model and create X_test before tuning."
#     )

# # ========== 用户可改：搜索策略与运行开关 ==========
# DRY_RUN = False               # True: 不训练，只打印组合/写最小日志
# SKIP_IF_LOGGED = True         # True: 如果日志里已有 grid_id 就跳过
# MAX_RUNS = None               # 例如 20；None 表示不限次数（但 heuristic 本身更省）
# TOP_K = 3                     # 记录 Top-K（默认 3）

# MAX_ROUNDS = 5                # 坐标搜索轮数上限（每轮会依次优化每个超参）
# NO_IMPROVE_TOL = 1e-6         # bal_acc 提升阈值（太小会抖动；太大可能停太早）

# LOG_PATH = ROOT / "experiments" / "xgb_rolling" / "grid_log_xgb_rolling_local.csv"
# TOPK_PATH = ROOT / "experiments" / "xgb_rolling" / "grid_top3_xgb_rolling_local.csv"
# BEST_OUT_PATH = ROOT / "experiments" / "best_params.json"

# # 候选值（每次只优化一个参数，其它固定）
# PARAM_CANDIDATES = {
#     "learning_rate": [0.005, 0.01, 0.02, 0.03],
#     "max_depth": [4, 5, 6, 7],
#     "min_child_weight": [1.0, 2.0, 4.0],
#     "subsample": [0.7, 0.8, 0.9],
#     "colsample_bytree": [0.7, 0.8, 0.9],
#     "reg_lambda": [0.5, 1.0, 2.0, 3.0],
#     "reg_alpha": [0.0, 0.5, 1.0, 2.0],
#     "gamma": [0.0, 0.5],
#     "scale_pos_weight": [2, 3, 5],
# }

# # 优化顺序（一般先学率/深度，再正则，再采样/类权重）
# SEARCH_ORDER = [
#     "learning_rate",
#     "max_depth",
#     "min_child_weight",
#     "subsample",
#     "colsample_bytree",
#     "reg_lambda",
#     "reg_alpha",
#     "gamma",
#     "scale_pos_weight",
# ]

# # 初始点（缺省会尽量从 xgb_params 里取；取不到就用候选列表的第一个）
# INITIAL_PARAMS = {
#     "learning_rate": float(xgb_params.get("learning_rate", PARAM_CANDIDATES["learning_rate"][0])),
#     "max_depth": int(xgb_params.get("max_depth", PARAM_CANDIDATES["max_depth"][0])),
#     "min_child_weight": float(xgb_params.get("min_child_weight", PARAM_CANDIDATES["min_child_weight"][0])),
#     "subsample": float(xgb_params.get("subsample", PARAM_CANDIDATES["subsample"][0])),
#     "colsample_bytree": float(xgb_params.get("colsample_bytree", PARAM_CANDIDATES["colsample_bytree"][0])),
#     "reg_lambda": float(xgb_params.get("reg_lambda", PARAM_CANDIDATES["reg_lambda"][0])),
#     "reg_alpha": float(xgb_params.get("reg_alpha", PARAM_CANDIDATES["reg_alpha"][0])),
#     "gamma": float(xgb_params.get("gamma", PARAM_CANDIDATES["gamma"][0])),
#     # 这个在 xgb_params 里通常不存在（我们在 train_params 里临时注入）
#     "scale_pos_weight": 2,
# }

# # ========== 工具函数 ==========
# def _hash_dict(d: dict) -> str:
#     payload = json.dumps(d, sort_keys=True, ensure_ascii=True, default=str).encode("utf-8")
#     return hashlib.md5(payload).hexdigest()[:8]


# def _append_csv(path: Path, row: dict) -> None:
#     path.parent.mkdir(parents=True, exist_ok=True)
#     needs_header = not path.exists()
#     with path.open("a", newline="") as f:
#         w = csv.DictWriter(f, fieldnames=list(row.keys()))
#         if needs_header:
#             w.writeheader()
#         w.writerow(row)


# def _load_logged_ids(path: Path) -> set[str]:
#     if not path.exists():
#         return set()
#     try:
#         df = pd.read_csv(path, usecols=["grid_id"])
#         return set(df["grid_id"].astype(str).tolist())
#     except Exception:
#         return set()


# def _sort_key(row: dict) -> tuple:
#     return (float(row.get("val_bal_acc", -1.0)), float(row.get("val_auc", -1.0)))


# def _better(a: dict | None, b: dict | None) -> dict | None:
#     if a is None:
#         return b
#     if b is None:
#         return a
#     return b if _sort_key(b) > _sort_key(a) else a


# def _update_topk(topk: list[dict], row: dict, k: int) -> list[dict]:
#     merged = topk + [row]
#     merged = sorted(merged, key=_sort_key, reverse=True)
#     out = []
#     seen = set()
#     for r in merged:
#         gid = str(r.get("grid_id"))
#         if gid in seen:
#             continue
#         seen.add(gid)
#         out.append(r)
#         if len(out) >= int(k):
#             break
#     return out


# def _read_json(path: Path) -> dict:
#     if not path.exists():
#         return {}
#     try:
#         return json.loads(path.read_text(encoding="utf-8"))
#     except Exception:
#         return {}


# def _write_best_params(topk_rows: list[dict]) -> None:
#     payload = _read_json(BEST_OUT_PATH)
#     now = datetime.utcnow().isoformat(timespec="seconds")
#     top_runs = []
#     for i, r in enumerate(topk_rows, start=1):
#         params = {k[2:]: r[k] for k in r.keys() if k.startswith("p_")}
#         top_runs.append({
#             "rank": int(i),
#             "grid_id": str(r.get("grid_id")),
#             "run_name": str(r.get("run_name")),
#             "val_bal_acc": float(r.get("val_bal_acc")),
#             "val_auc": float(r.get("val_auc")),
#             "thr": float(r.get("thr")),
#             "best_n_estimators": int(r.get("best_n_estimators")),
#             "params": params,
#         })
#     best = top_runs[0] if top_runs else None
#     payload["xgb_rolling"] = {
#         "metric": "val_bal_acc",
#         "updated_at_utc": now,
#         "run_name": None if best is None else best["run_name"],
#         "grid_id": None if best is None else best["grid_id"],
#         "val_bal_acc": None if best is None else best["val_bal_acc"],
#         "val_auc": None if best is None else best["val_auc"],
#         "thr": None if best is None else best["thr"],
#         "top_runs": top_runs,
#     }
#     BEST_OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
#     BEST_OUT_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


# def run_one_xgb(params_override: dict) -> dict:
#     # 1) 训练（带 early stopping）
#     train_params = dict(xgb_params)
#     train_params.update({k: v for k, v in params_override.items() if k != "scale_pos_weight"})
#     train_params["scale_pos_weight"] = params_override.get("scale_pos_weight", 2)

#     model = XGBClassifier(**train_params, callbacks=XGB_CALLBACKS)
#     model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)

#     val_proba = model.predict_proba(X_val)[:, 1]
#     val_auc = float(roc_auc_score(y_val, val_proba))

#     # 2) 阈值（balanced accuracy）
#     thresholds = np.linspace(0.01, 0.99, 99)
#     bal_scores = [balanced_accuracy_score(y_val, (val_proba >= t).astype(int)) for t in thresholds]
#     best_idx = int(np.argmax(bal_scores))
#     best_thr = float(thresholds[best_idx])
#     best_bal = float(bal_scores[best_idx])

#     # 3) 训练最终模型（train+val）
#     best_iteration = getattr(model, "best_iteration", None)
#     best_n_estimators = int(model.get_params()["n_estimators"]) if best_iteration is None else int(best_iteration) + 1

#     X_all = pd.concat([X_train, X_val], axis=0)
#     y_all = pd.concat([y_train, y_val], axis=0)
#     final_params = dict(train_params)
#     final_params["n_estimators"] = int(best_n_estimators)
#     final_model = XGBClassifier(**final_params)
#     final_model.fit(X_all, y_all, verbose=False)

#     # 4) test 预测 + 提交文件
#     test_proba = final_model.predict_proba(X_test)[:, 1]
#     pred = pd.DataFrame({"id": X_test.index.astype(int), "proba": test_proba.astype(float)})
#     example = pd.read_csv(EXAMPLE_SUB)
#     out = example[["id"]].merge(pred, on="id", how="left")
#     out["proba"] = out["proba"].fillna(0.0)
#     out["target"] = (out["proba"] >= float(best_thr)).astype(int)
#     out = out[["id", "target"]]
#     test_pos_rate = float(out["target"].mean())

#     # 5) 保存 artifacts（供 ensemble / 手动上传 Kaggle 使用）
#     grid_id = _hash_dict(params_override)
#     run_name = (
#         f"xgb_rolling_grid_cut{CUTOFF_START}-{CUTOFF_END}_{CUTOFF_FREQ}_"
#         f"sub{SUBMISSION_CUTOFF}_h{HORIZON_DAYS}_lb{LOOKBACK_DAYS}_g{grid_id}"
#     )
#     art_dir = ROOT / "experiments" / "xgb_rolling" / "artifacts" / run_name
#     sub_dir = ROOT / "experiments" / "xgb_rolling" / "submissions"
#     art_dir.mkdir(parents=True, exist_ok=True)
#     sub_dir.mkdir(parents=True, exist_ok=True)

#     model_path = art_dir / "model.json"
#     final_model.save_model(model_path)
#     (art_dir / "feature_columns.json").write_text(
#         json.dumps([str(c) for c in X_train.columns], ensure_ascii=False, indent=2),
#         encoding="utf-8",
#     )
#     (art_dir / "final_params.json").write_text(json.dumps(final_params, ensure_ascii=False, indent=2), encoding="utf-8")
#     (art_dir / "grid_params.json").write_text(json.dumps(params_override, ensure_ascii=False, indent=2), encoding="utf-8")
#     (art_dir / "train_meta.json").write_text(
#         json.dumps(
#             {
#                 "val_auc": val_auc,
#                 "best_thr": best_thr,
#                 "best_bal": best_bal,
#                 "best_n_estimators": int(best_n_estimators),
#             },
#             ensure_ascii=False,
#             indent=2,
#         ),
#         encoding="utf-8",
#     )

#     sub_path = sub_dir / f"{run_name}.csv"
#     out.to_csv(sub_path, index=False)

#     result = {
#         "logged_at_utc": datetime.utcnow().isoformat(timespec="seconds"),
#         "grid_id": grid_id,
#         "run_name": run_name,
#         "val_auc": val_auc,
#         "val_bal_acc": best_bal,
#         "thr": best_thr,
#         "test_pos_rate": test_pos_rate,
#         "best_n_estimators": int(best_n_estimators),
#     }
#     for k, v in params_override.items():
#         result[f"p_{k}"] = v
#     return result


# def _evaluate(params_override: dict, *, topk: list[dict], logged_ids: set[str]) -> tuple[dict | None, list[dict]]:
#     grid_id = _hash_dict(params_override)
#     if SKIP_IF_LOGGED and grid_id in logged_ids:
#         return None, topk

#     print(f"\n=== eval grid_id={grid_id} params={params_override}")
#     if DRY_RUN:
#         row = {"logged_at_utc": datetime.utcnow().isoformat(timespec="seconds"), "grid_id": grid_id}
#         for k, v in params_override.items():
#             row[f"p_{k}"] = v
#     else:
#         row = run_one_xgb(params_override)
#         topk = _update_topk(topk, row, TOP_K)
#         pd.DataFrame(topk).sort_values(["val_bal_acc", "val_auc"], ascending=False).to_csv(TOPK_PATH, index=False)
#         _write_best_params(topk)

#     _append_csv(LOG_PATH, row)
#     logged_ids.add(grid_id)
#     return row, topk


# def _params_in_order(params: dict) -> dict:
#     return {k: params[k] for k in SEARCH_ORDER if k in params}


# def coordinate_search() -> tuple[list[dict], dict]:
#     logged_ids = _load_logged_ids(LOG_PATH) if SKIP_IF_LOGGED else set()
#     topk: list[dict] = []

#     current = dict(INITIAL_PARAMS)
#     best_overall: dict | None = None
#     ran = 0

#     print("search_order:", SEARCH_ORDER)
#     print("initial:", _params_in_order(current))
#     print("already logged:", len(logged_ids))

#     # baseline
#     row0, topk = _evaluate(current, topk=topk, logged_ids=logged_ids)
#     if row0 is not None:
#         best_overall = _better(best_overall, row0)
#         ran += 1

#     for rnd in range(1, int(MAX_ROUNDS) + 1):
#         if MAX_RUNS is not None and ran >= int(MAX_RUNS):
#             break

#         print(f"\n===== Round {rnd}/{MAX_ROUNDS}  current={_params_in_order(current)}")
#         improved_any = False

#         for p in SEARCH_ORDER:
#             if MAX_RUNS is not None and ran >= int(MAX_RUNS):
#                 break
#             if p not in PARAM_CANDIDATES:
#                 continue

#             base_row = best_overall
#             base_key = _sort_key(base_row) if base_row is not None else (-1.0, -1.0)

#             best_for_p_row: dict | None = base_row
#             best_for_p_val = current.get(p)

#             candidates = list(PARAM_CANDIDATES[p])
#             if best_for_p_val in candidates:
#                 candidates = [best_for_p_val] + [v for v in candidates if v != best_for_p_val]

#             print(f"\n-- Optimize {p}: candidates={candidates}")
#             for v in candidates:
#                 if MAX_RUNS is not None and ran >= int(MAX_RUNS):
#                     break
#                 trial = dict(current)
#                 trial[p] = v

#                 row, topk = _evaluate(trial, topk=topk, logged_ids=logged_ids)
#                 if row is None:
#                     continue
#                 ran += 1
#                 best_overall = _better(best_overall, row)
#                 best_for_p_row = _better(best_for_p_row, row)
#                 if best_for_p_row is row:
#                     best_for_p_val = v

#             new_key = _sort_key(best_for_p_row) if best_for_p_row is not None else (-1.0, -1.0)
#             if (new_key[0] > base_key[0] + float(NO_IMPROVE_TOL)) or (new_key[0] == base_key[0] and new_key[1] > base_key[1]):
#                 current[p] = best_for_p_val
#                 improved_any = True
#                 print(f"accept {p}={best_for_p_val}  base={base_key} -> new={new_key}")
#             else:
#                 print(f"keep   {p}={current.get(p)}  best_in_param={best_for_p_val}  base={base_key} new={new_key}")

#         if not improved_any:
#             print(f"\nNo improvement in round {rnd}; stop.")
#             break

#     print("\nDone. ran=", ran)
#     if topk:
#         best = topk[0]
#         print("Top-1:", {"val_bal_acc": best.get("val_bal_acc"), "val_auc": best.get("val_auc"), "thr": best.get("thr"), "grid_id": best.get("grid_id"), "run_name": best.get("run_name")})
#     else:
#         print("Top-1: None (maybe DRY_RUN or no runs executed)")

#     return topk, current


# topk, best_params = coordinate_search()
# best_params